# Лекция 2: От линейни модели до невронни мрежи
**От LLM до агенти** — Лекция 2 от 12

## Рекап от Лекция 1

- **Езиковият модел предсказва следващия токен** — видяхме GPT-2 в действие
- **Построихме n-gram модели** (bigram, trigram) — предсказване чрез броене на честоти
- **One-hot вектори нямат понятие за сходство** — всички символи са еднакво далеч един от друг
- **Embeddings на GPT-2**: сходни думи → близки вектори в пространството
- **Днес**: как моделът *научава* тези вектори — и как изобщо се учи нещо?

## Какво ще разгледаме днес

Преходът от *броене* към *учене*: строим параметризирани модели, обучени с градиентно спускане.
Същата задача като Лекция 1 — предсказване на следващ символ — но вместо да броим, оптимизираме тегла.

### Блокове

1. **Преговор и мотивация** [Обяснение] — Броене срещу учене; какво е параметър?
2. **Представяне на символи и линеен модел** [Демо] — One-hot кодиране, матрица W, разглеждане на един пример стъпка по стъпка
3. **Loss функция и градиент** [Обяснение] — -log(P) като мярка за грешка, градиент и правило за обновяване
4. **Обучение с градиентно спускане** [Демо] — Тренировъчен цикъл, кривата на loss, генерация на текст, сравнение с броене
5. **Разширяване на контекста** [Демо] — Конкатениране на one-hot вектори за повече символи
6. **От линейност до невронни мрежи** [Обяснение] — Защо линейните слоеве се сливат; sigmoid и нелинейност; autograd
7. **Обучаваме невронна мрежа** [Демо] — CharNN с скрит слой, сравнение с линеен модел
8. **Embedding слоеве** [Демо] — nn.Embedding, BigramNN, визуализация на 2D вектори
9. **Обобщение и дискусия** [Дискусия] — Въпроси за размисъл, поглед напред

---
## Блок 1: Преговор и мотивация [Обяснение]

**Цел:** Да осмислим защо преминаваме от броене към учене — и какво точно означава „моделът учи".

### Два подхода за предсказване на следващия символ

**Подход 1 — Броене (Лекция 1):**
Преминаваме през целия корпус, броим колко пъти след символ „а" идва „б", нормализираме → вероятностна таблица.

**Подход 2 — Учене (днес):**
Инициализираме матрица от тегла W на случаен принцип, след това итеративно я коригираме,
така че тя да дава по-добри предсказвания за следващия символ.

**Три проблема с броенето:**

1. **Не генерализира** — ако никога не сме видели „зх" в корпуса, вероятността е 0 (или изисква изкуствено изглаждане)
2. **Не се мащабира** — таблицата за 5-gram при речник V=197 изисква V⁵ ≈ 3×10¹¹ записи
3. **Не учи представяния** — bigram таблицата не знае, че „а", „е", „и" са гласни; всеки символ е изолиран

### Какво е параметър (или тегла) на езиковия модел?

**Параметрите** са числата, които моделът *настройва* по време на обучение.
За линеен модел това са матрицата на теглата **W** и вектора на отместването **b**.

Процесът на обучение в четири стъпки:

1. **Случайна инициализация** — W и b получават произволни числа
2. **Предсказване** — подаваме входа, получаваме изходни вероятности
3. **Измерване на грешката** — изчисляваме колко сгрешихме (loss функция)
4. **Коригиране на теглата** — местим W и b в посоката, която намалява грешката

После повтаряме стъпки 2-4 хиляди пъти, докато моделът стане достатъчно добър.

### Нашият инструмент: PyTorch

 ![pytorch](./images/pytorch.png)

За да реализираме тези стъпки, ще използваме **PyTorch** — библиотеката за машинно учене, която стои зад повечето съвременни езикови модели (GPT, LLaMA, Gemma и др.).

PyTorch ни дава пет ключови неща:

| Модул | Какво прави | Пример |
|-------|-------------|--------|
| **`torch`** | Тензори — многомерни масиви като NumPy, но с поддръжка на GPU и автоматично диференциране | `torch.randn(3, 4)` |
| **`torch.nn`** | Готови градивни блокове за невронни мрежи — слоеве, модели | `nn.Linear(10, 5)`, `nn.Embedding(100, 8)` |
| **`torch.nn.functional` (F)** | Функции без състояние — активации, loss функции, кодиране | `F.cross_entropy(...)`, `F.softmax(...)` |
| **`torch.optim`** | Оптимизатори — алгоритми за обновяване на теглата | `optim.SGD(...)`, `optim.Adam(...)` |
| **autograd** | Автоматично изчисляване на градиенти — сърцето на обучението | `requires_grad=True`, `.backward()`, `.grad` |

Днес ще използваме всичките пет. Нека видим как изглеждат на практика.

In [ ]:
import torch
import torch.nn.functional as F

# Тензор — основната структура в PyTorch (като NumPy array)
x = torch.tensor([1.0, 2.0, 3.0])
print(f"Вектор:  {x}  (shape: {x.shape}, dtype: {x.dtype})")

In [ ]:
# Матрица с произволни стойности
M = torch.randn(3, 4)
N = torch.randn(4, 1)
print(f"Матрица: shape {M.shape}")

In [ ]:
# Матрично умножение с @
result = x @ M
print(f"x @ M:   {result}  (shape: {result.shape})")

In [ ]:
# requires_grad=True казва на PyTorch: „следи операциите, за да може после да изчислиш градиенти"
w = torch.randn(3, requires_grad=True)
print(f"\nw = {w.data.numpy().round(3)}, requires_grad={w.requires_grad}")
print("→ PyTorch ще записва всяка операция с w, за да може после .backward() да изчисли ∂L/∂w")

---
## Блок 2: Представяне на символи и линеен модел [Демо]

**Цел:** Да заредим данните, да представим символите като вектори и да построим първия си параметризиран модел.

In [ ]:
from tools import (
    load_corpus, preprocess, build_vocab, train_test_split,
    build_context_dataset, generate, generate_context, encode_context,
    evaluate_linear, evaluate_context_linear, evaluate_nn,
)

In [ ]:
raw_text = load_corpus("../data/ivan_vazov.txt")
print(f"Суров корпус: {len(raw_text):,} символа, {len(set(raw_text))} уникални символа")

In [ ]:
text = preprocess(raw_text, lowercase=True, top_n=40)
print(f"След предобработка: {len(text):,} символа, {len(set(text))} уникални символа")

In [ ]:
chars, char_to_idx, idx_to_char = build_vocab(text)
vocab_size = len(chars)
print(f"Речник ({vocab_size} символа):")
print(repr("".join(chars)))

In [ ]:
train_text, test_text = train_test_split(text)
print(f"Train: {len(train_text):,} символа")
print(f"Test:  {len(test_text):,} символа")

In [ ]:
def build_bigram_dataset(text):
    xs, ys = [], []
    for i in range(len(text) - 1):
        xs.append(char_to_idx[text[i]])
        ys.append(char_to_idx[text[i + 1]])
    return torch.tensor(xs), torch.tensor(ys)

X_train, Y_train = build_bigram_dataset(train_text)
X_test, Y_test = build_bigram_dataset(test_text)

print(f"Training: {len(X_train):,} двойки")
print(f"Test: {len(X_test):,} двойки")

In [ ]:
print("Първи 5 примера (вход → изход):")
print(f"{'Индекс вход':>12} {'Символ вход':>12} {'Индекс изход':>13} {'Символ изход':>13}")
print("-" * 55)
for i in range(5):
    xi = X_train[i].item()
    yi = Y_train[i].item()
    print(f"{xi:>12} {repr(idx_to_char[xi]):>12} {yi:>13} {repr(idx_to_char[yi]):>13}")

### Линеен модел

**Вход:** one-hot вектор **x** с дължина V (всички нули, освен позицията на текущия символ).

**Изходни логити:**

$$\text{logits} = \mathbf{x} \cdot W + \mathbf{b}$$

където $W \in \mathbb{R}^{V \times V}$ е матрицата на теглата и $\mathbf{b} \in \mathbb{R}^{V}$ е вектор на отместването.

**Вероятности** (softmax):

$$P(\text{следващ} = j \mid \text{текущ} = i) = \frac{e^{\text{logits}_j}}{\sum_k e^{\text{logits}_k}}$$

![linear](./images/linear.png)

In [ ]:
torch.manual_seed(42)
W = torch.randn(vocab_size, vocab_size, requires_grad=True)
b = torch.zeros(vocab_size, requires_grad=True)
print(f"W: {W.shape} = {W.numel():,} тегла")
print(f"b: {b.shape} = {b.numel()} стойности")
print(f"Общо параметри: {W.numel() + b.numel():,}")

### Разглеждаме ЕДИН пример стъпка по стъпка

Преди да тренираме, нека проследим какво се случва с един конкретен пример.
Ще вземем първата двойка от тренировъчния набор и ще я прекараме ръчно през модела.
Това прави абстрактното конкретно.

In [ ]:
# Стъпка 1: вземаме входния и целевия символ
input_idx = X_train[0].item()
target_idx = Y_train[0].item()
input_char = idx_to_char[input_idx]
target_char = idx_to_char[target_idx]
print(f"Стъпка 1 | Вход: {repr(input_char)} (индекс {input_idx}) → Цел: {repr(target_char)} (индекс {target_idx})")

In [ ]:
# Стъпка 2: one-hot кодиране
one_hot = F.one_hot(torch.tensor(input_idx), num_classes=vocab_size).float()
nonzero_pos = one_hot.nonzero().item()
print(f"Стъпка 2 | One-hot вектор: форма {list(one_hot.shape)}, ненулева позиция: {nonzero_pos}")

In [ ]:
# Стъпка 3: логити = x @ W + b
with torch.no_grad():
    logits = one_hot @ W + b
print(f"Стъпка 3 | Логити: форма {list(logits.shape)}, първи 5: {logits[:5].numpy().round(3)}")

In [ ]:
# Стъпка 4: softmax → вероятности
probs = torch.softmax(logits, dim=-1)
print(f"Стъпка 4 | Сума на вероятностите: {probs.sum().item():.6f} (трябва да е 1.0)")

In [ ]:
# Стъпка 5: вероятност за верния символ
p_correct = probs[target_idx].item()
random_baseline = 1 / vocab_size
print(f"Стъпка 5 | P('{target_char}') = {p_correct:.6f}")
print(f"          Случайно гадаене: {random_baseline:.6f} (1/{vocab_size})")
print(f"          Модел {'по-добър' if p_correct > random_baseline else 'по-лош'} от случайното!")

---
## Блок 3: Loss функция и градиент [Обяснение]

**Цел:** Да разберем как измерваме грешката на модела и как изчисляваме посоката за подобрение.

### Мотивация: имаме нужда от число, което да минимизираме

Искаме моделът да дава **висока вероятност** на верния следващ символ.
Но как превръщаме „вероятността е висока" в число, подходящо за оптимизация?

Разгледайте три случая:

| Вероятност на верния символ | $-\log(P)$ | Оценка |
|-----------------------------|-------------|--------|
| $P = 0.01$ | $-\log(0.01) \approx 4.6$ | Много лошо! |
| $P = 0.50$ | $-\log(0.50) \approx 0.69$ | Средно |
| $P = 0.99$ | $-\log(0.99) \approx 0.01$ | Отлично! |

**Идеята:** $-\log(P)$ е голямо когато предсказваме зле и малко когато предсказваме добре.
Затова го използваме като **loss функция** — нещо, което искаме да *минимизираме*.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

P = np.linspace(0.01, 0.99, 300)
loss_vals = -np.log(P)

plt.figure(figsize=(7, 4))
plt.plot(P, loss_vals, color='steelblue', linewidth=2)

# Три характерни точки
for p_mark, label in [(0.01, "P=0.01\nloss≈4.6"), (0.50, "P=0.50\nloss≈0.69"), (0.99, "P=0.99\nloss≈0.01")]:
    plt.scatter([p_mark], [-np.log(p_mark)], color='red', s=80, zorder=5)
    plt.annotate(label, (p_mark, -np.log(p_mark)),
                 textcoords="offset points", xytext=(8, 5), fontsize=9)

plt.xlabel("Вероятност P (за верния символ)", fontsize=11)
plt.ylabel("-log(P)", fontsize=11)
plt.title("Loss функция: −log(P)", fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Cross-entropy loss

**Cross-entropy** е средното $-\log(P)$ по всички примери в набора:

$$L = -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i \mid x_i)$$

където $y_i$ е верният следващ символ за пример $i$.

**Какъв loss очакваме при случайни тегла?**

При случаен модел, $P \approx \frac{1}{V}$, следователно:

$$L_{\text{случайно}} \approx -\log\left(\frac{1}{V}\right) = \log(V)$$

При $V = 40$: $\log(40) \approx 3.69$. Това е нашата базова линия — ако моделът не е по-добър от нея, не е научил нищо.

In [ ]:
import math

# Изчисляваме loss върху малка партида от 32 примера
batch_size = 32
ix = torch.arange(batch_size)
x_batch = F.one_hot(X_train[ix], num_classes=vocab_size).float()

with torch.no_grad():
    logits = x_batch @ W + b
    loss = F.cross_entropy(logits, Y_train[ix])

random_baseline_loss = math.log(vocab_size)
print(f"Loss на текущия модел (случайни тегла): {loss.item():.4f}")
print(f"Очакван loss при случайно гадаене:       {random_baseline_loss:.4f}  (log({vocab_size}))")
print(f"Разлика: {loss.item() - random_baseline_loss:+.4f} (близо до 0 — моделът е почти случаен)")

### Градиент: посока за подобрение

**Градиентът** $\nabla_W L$ е вектор (матрица), който казва:
„ако увеличиш тегло $w_{ij}$ с малко $\epsilon$, loss ще се промени с $\frac{\partial L}{\partial w_{ij}} \cdot \epsilon$."

**Правило за обновяване:**

$$W \leftarrow W - \eta \cdot \frac{\partial L}{\partial W}$$

$$b \leftarrow b - \eta \cdot \frac{\partial L}{\partial b}$$

Тук $\eta$ е **learning rate** — колко голяма стъпка правим.
Малко $\eta$ → бавна, сигурна конвергенция. Голямо $\eta$ → бързо, но рискуваме да „прескочим" минимума.

In [ ]:
# Изчисляваме градиенти върху малка партида
ix = torch.arange(32)
x_batch = F.one_hot(X_train[ix], num_classes=vocab_size).float()
logits = x_batch @ W + b
loss = F.cross_entropy(logits, Y_train[ix])

loss.backward()  # PyTorch изчислява ∂L/∂W и ∂L/∂b автоматично

print(f"W.grad форма:  {W.grad.shape}")
print(f"b.grad форма:  {b.grad.shape}")
print(f"Средна абс. стойност ∂L/∂W: {W.grad.abs().mean().item():.6f}")
print(f"Средна абс. стойност ∂L/∂b: {b.grad.abs().mean().item():.6f}")
print()
print("Интерпретация: всеки параметър знае в коя посока трябва да се придвижи,")
print("за да намали грешката на модела.")

---
## Блок 4: Обучение с градиентно спускане [Демо]

**Цел:** Да обучим линейния модел и да видим как loss намалява с времето.

![gradient-decent](./images/gradient_decent.png)

In [ ]:
torch.manual_seed(42)
W = torch.randn(vocab_size, vocab_size, requires_grad=True)
b = torch.zeros(vocab_size, requires_grad=True)

learning_rate = 1.0
losses = []

for step in range(400):
    ix = torch.randint(0, len(X_train), (4096,))
    x_batch = F.one_hot(X_train[ix], num_classes=vocab_size).float()

    logits = x_batch @ W + b
    loss = F.cross_entropy(logits, Y_train[ix])

    W.grad = None
    b.grad = None
    loss.backward()

    with torch.no_grad():
        W -= learning_rate * W.grad
        b -= learning_rate * b.grad

    losses.append(loss.item())
    if step % 50 == 0:
        print(f"Стъпка {step:3d}: loss = {loss.item():.4f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses, color='steelblue', linewidth=1.5, label='Loss (линеен модел)')
plt.axhline(math.log(vocab_size), color='red', linestyle='--',
            label=f'Случайно гадаене (log({vocab_size}) ≈ {math.log(vocab_size):.2f})')
plt.xlabel("Стъпка на обучение")
plt.ylabel("Cross-entropy loss")
plt.title("Loss по време на обучение (линеен модел)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Оценяване на тест набора
with torch.no_grad():
    batch_size = 4096
    total_loss = 0.0
    correct = 0
    for i in range(0, len(X_test), batch_size):
        xb = F.one_hot(X_test[i:i+batch_size], num_classes=vocab_size).float()
        yb = Y_test[i:i+batch_size]
        logits = xb @ W + b
        total_loss += F.cross_entropy(logits, yb, reduction='sum').item()
        correct += (logits.argmax(dim=1) == yb).sum().item()

    test_loss = total_loss / len(X_test)
    test_acc = correct / len(X_test)

print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.1%}")
print(f"(случайно гадаене: {1/vocab_size:.1%})")

In [ ]:
print("=== Генериран текст (линеен модел) ===")
print(generate(lambda x: x @ W + b, char_to_idx, idx_to_char, one_hot=True))

In [ ]:
# Counting bigram от тренировъчните данни (за сравнение)
bigram_counts = torch.zeros(vocab_size, vocab_size)
for i in range(len(train_text) - 1):
    bigram_counts[char_to_idx[train_text[i]], char_to_idx[train_text[i + 1]]] += 1

# Без изглаждане (за визуализация)
bigram_probs = bigram_counts / bigram_counts.sum(dim=1, keepdim=True).clamp(min=1)

# С изглаждане (за оценяване)
bigram_probs_smooth = (bigram_counts + 1) / (bigram_counts.sum(dim=1, keepdim=True) + vocab_size)

# Оценяване на counting модела
counting_preds = bigram_probs_smooth.argmax(dim=1)
counting_acc = (counting_preds[X_test] == Y_test).float().mean().item()
counting_loss = -torch.log(bigram_probs_smooth[X_test, Y_test]).mean().item()

print(f"Counting bigram — loss: {counting_loss:.4f}, accuracy: {counting_acc:.1%}")
print(f"Линеен модел    — loss: {test_loss:.4f}, accuracy: {test_acc:.1%}")

### Моделът преоткри броенето!

Нека сравним какво е *научил* линейният модел с таблицата от броене.

Ако линейният модел е обучен правилно, матрицата `softmax(W + b)` трябва да прилича
на нормализираната bigram таблица от Лекция 1.
Проверяваме това чрез топлинни карти и корелация.

In [ ]:
# Научено разпределение
with torch.no_grad():
    learned_probs = F.softmax(W + b, dim=1)

# 15 чести символа за четима топлинна карта
common_chars = [' ', 'а', 'б', 'в', 'г', 'д', 'е', 'и', 'к', 'н', 'о', 'р', 'с', 'т', 'я']
indices = [char_to_idx[c] for c in common_chars]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, probs, title in [
    (axes[0], bigram_probs, "Counting bigram (Лекция 1)"),
    (axes[1], learned_probs, "Линеен модел (softmax(W+b))")
]:
    sub = probs[indices][:, indices].numpy()
    im = ax.imshow(sub, cmap='Blues')
    ax.set_xticks(range(len(common_chars)))
    ax.set_yticks(range(len(common_chars)))
    ax.set_xticklabels([repr(c) for c in common_chars], fontsize=8)
    ax.set_yticklabels([repr(c) for c in common_chars], fontsize=8)
    ax.set_title(title, fontsize=12)
    plt.colorbar(im, ax=ax)

plt.suptitle("Моделът преоткри броенето!", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Корелация между двете разпределения
corr = torch.corrcoef(torch.stack([
    learned_probs.flatten(),
    bigram_probs.flatten()
]))[0, 1]
print(f"Корелация между двете разпределения: {corr:.4f}")

### Интерпретация

**Линейният модел сходира към приблизително същото решение като броенето.**
Оптимизацията откри нещо, което можеше да се намери и по-просто — но я намери сама, само чрез градиенти.

Това е мощна валидация: методът работи.

**Но също така е тавана на линейния модел.**
Колкото и дълго да го тренираме, той не може да надмине counting bigram —
защото има точно толкова параметри и точно толкова изразителна сила.

> За да победим броенето, трябва нещо, което броенето не може:
> **повече контекст** и **нелинейност**.

---
## Блок 5: Разширяване на контекста [Демо]

**Цел:** Да видим какво се случва, ако подаваме повече символи на линейния модел — и защо това не е достатъчно.


In [ ]:
print(f"encode_context: контекст 3 → вход с размер {3 * vocab_size} (3 one-hot вектора конкатенирани)")

In [ ]:
results = {}
for ctx in [1, 2, 3]:
    X_ctx_train, Y_ctx_train = build_context_dataset(train_text, ctx, char_to_idx)
    X_ctx_test, Y_ctx_test = build_context_dataset(test_text, ctx, char_to_idx)

    input_dim = ctx * vocab_size
    torch.manual_seed(42)
    W_ctx = torch.randn(input_dim, vocab_size, requires_grad=True)
    b_ctx = torch.zeros(vocab_size, requires_grad=True)

    for step in range(1000):
        ix = torch.randint(0, len(X_ctx_train), (4096,))
        x_batch = encode_context(X_ctx_train[ix], vocab_size)
        logits = x_batch @ W_ctx + b_ctx
        loss = F.cross_entropy(logits, Y_ctx_train[ix])
        W_ctx.grad = None
        b_ctx.grad = None
        loss.backward()
        with torch.no_grad():
            W_ctx -= 10.0 * W_ctx.grad
            b_ctx -= 10.0 * b_ctx.grad

    t_loss, t_acc = evaluate_context_linear(W_ctx, b_ctx, X_ctx_test, Y_ctx_test, vocab_size)
    n_params = W_ctx.numel() + b_ctx.numel()
    results[ctx] = (n_params, t_loss, t_acc)
    print(f"Контекст {ctx}: {n_params:>7,} параметъра | loss = {t_loss:.4f} | accuracy = {t_acc:.1%}")

In [ ]:
print()
print(f"{'Контекст':<10} {'Параметри':>12} {'Test loss':>10} {'Accuracy':>10}")
print("-" * 46)
for ctx, (n_params, t_loss, t_acc) in results.items():
    print(f"bigram ({ctx}){'':<3} {n_params:>12,} {t_loss:>10.4f} {t_acc:>10.1%}")

### Наблюдение

Точността расте с контекста, но достига таван — линейният модел е фундаментално ограничен.

**Защо?** Конкатенирането на one-hot вектори е разточително:
за контекст 3 входният вектор има $3 \times 40 = 120$ елемента, почти всички нули.

**Ами ако вместо one-hot използваме по-кратки, *научени* вектори?**

Тогава вместо 120 нули, имаме само 6 числа (напр. 3 вектора × 2 измерения).
И тези числа не са произволни — те са научени така, че сходни символи да имат близки вектори.

Но дори и с one-hot — **линейният модел удари таван**.
Повече контекст помага, но линейната функция не може да улови сложните зависимости между символите.

→ В Блок 7 ще видим как **нелинейността** чупи този таван. А в Блок 8 — как **embeddings** го правят ефективно.

---
## Блок 6: От линейност до невронни мрежи [Обяснение]

**Цел:** Да разберем защо добавянето на повече линейни слоеве не помага — и как нелинейността решава проблема.

![ff](./images/feed_forward.png)

### Два линейни слоя = един линеен слой

Нека имаме два линейни слоя един след друг:

$$h = x \cdot W_1, \quad y = h \cdot W_2 = x \cdot W_1 \cdot W_2 = x \cdot W_3$$

Умножението $W_1 \cdot W_2$ дава нова матрица $W_3$ — пак линейна трансформация.

**Колкото и слоя да наредим, резултатът е еквивалентен на един линеен слой.**
Добавянето на дълбочина без нелинейност не ни дава нищо допълнително.

In [ ]:
torch.manual_seed(0)
W1 = torch.randn(3, 4)
W2 = torch.randn(4, 3)
x = torch.randn(1, 3)

hidden = x @ W1
output_two = hidden @ W2

W_combined = W1 @ W2
output_one = x @ W_combined

print(f"Два слоя:        {output_two.data}")
print(f"Един комбиниран: {output_one.data}")
print()
print("Идентични! Добавянето на слоеве без нелинейност не помага.")

### Решение: нелинейна активационна функция

Добавяме **активационна функция** между слоевете:

$$h = \sigma(x \cdot W_1), \quad y = h \cdot W_2$$

Функцията **sigmoid** $\sigma(z) = \frac{1}{1 + e^{-z}}$ смачква всяко реално число в интервала $(0, 1)$.

**Неврон** = линейна комбинация на входовете + активационна функция:

$$\text{output} = \sigma\!\left(\sum_i w_i x_i + b\right)$$

Sigmoid е гладка, диференцируема навсякъде — важно за обратното разпространение на сигнала.

In [ ]:
hidden_linear = x @ W1                  # без активация
hidden_sigmoid = torch.sigmoid(x @ W1)  # със sigmoid

output_linear = hidden_linear @ W2
output_nonlinear = hidden_sigmoid @ W2

print(f"Без sigmoid:  {output_linear.data}")
print(f"Със sigmoid:  {output_nonlinear.data}")
print()
print("Различни! Sigmoid-ът прави модела нелинеен и изразителен.")

In [ ]:
x_vals = np.linspace(-6, 6, 300)
sigmoid_vals = 1 / (1 + np.exp(-x_vals))
tanh_vals = np.tanh(x_vals)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(x_vals, sigmoid_vals, color='steelblue', linewidth=2)
axes[0].axhline(0, color='gray', linewidth=0.5)
axes[0].axvline(0, color='gray', linewidth=0.5)
axes[0].set_title("Sigmoid: σ(z) = 1 / (1 + e⁻ᶻ)", fontsize=12)
axes[0].set_xlabel("z")
axes[0].set_ylabel("σ(z)")
axes[0].set_ylim(-0.1, 1.1)
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_vals, tanh_vals, color='darkorange', linewidth=2)
axes[1].axhline(0, color='gray', linewidth=0.5)
axes[1].axvline(0, color='gray', linewidth=0.5)
axes[1].set_title("Tanh: tanh(z)", fontsize=12)
axes[1].set_xlabel("z")
axes[1].set_ylabel("tanh(z)")
axes[1].set_ylim(-1.1, 1.1)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Нелинейни активационни функции", fontsize=13)
plt.tight_layout()
plt.show()

### Обратно разпространение на сигнала (backpropagation)

Помните `requires_grad=True` от Блок 2? Ето какво PyTorch правеше зад кулисите — записваше всяка операция, за да може сега да изчисли как всяко тегло влияе на крайната грешка.

**Верижното правило** (chain rule):

$$\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial h} \cdot \frac{\partial h}{\partial W_1}$$

PyTorch прави това автоматично чрез **autograd**:
- Пазим история на всички операции (`requires_grad=True`)
- При `loss.backward()` — градиентите се изчисляват назад по веригата
- Всяко тегло получава `grad` — своята „отговорност" за грешката

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b_val = torch.tensor(3.0, requires_grad=True)
y = a * b_val + a ** 2
print(f"y = a·b + a² = {a.item()}·{b_val.item()} + {a.item()}² = {y.item()}")

y.backward()
print(f"∂y/∂a = b + 2a = {b_val.item()} + 2·{a.item()} = {a.grad.item()}")
print(f"∂y/∂b = a      = {a.item()} = {b_val.grad.item()}")
print()
print("PyTorch изчислява тези производни автоматично.")

---
## Блок 7: Обучаваме невронна мрежа [Демо]

**Цел:** Да построим невронна мрежа с един скрит слой и да покажем, че нелинейността чупи тавана на линейния модел.

**Стратегия:** Използваме същия контекст=3 от Блок 5, но добавяме скрит слой с нелинейна активация.

In [ ]:
import torch.nn as nn

# Подготовка: данни с контекст=3 (същите като Блок 5)
context_size = 3
X_ctx3_train, Y_ctx3_train = build_context_dataset(train_text, context_size, char_to_idx)
X_ctx3_test, Y_ctx3_test = build_context_dataset(test_text, context_size, char_to_idx)
print(f"Контекст={context_size}: {len(X_ctx3_train):,} тренировъчни, {len(X_ctx3_test):,} тест примера")
print(f"Вход: {context_size} символа → one-hot размер: {context_size * vocab_size}")

In [ ]:
class CharContextNN(nn.Module):
    """Невронна мрежа с контекстен прозорец: one-hot вход → скрит слой → изход."""
    def __init__(self, vocab_size, context_size=3, hidden_size=128):
        super().__init__()
        self.vocab_size = vocab_size
        self.layer1 = nn.Linear(context_size * vocab_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        # x: (B, context_size * vocab_size) — конкатенирани one-hot вектори
        h = torch.sigmoid(self.layer1(x))
        return self.layer2(h)

model_nn = CharContextNN(vocab_size, context_size)
total_params = sum(p.numel() for p in model_nn.parameters())
print(f"CharContextNN: {total_params:,} параметъра (вход={context_size}×{vocab_size}={context_size*vocab_size}, скрит=128)")
print(model_nn)

In [ ]:
torch.manual_seed(42)
model_nn = CharContextNN(vocab_size, context_size)
optimizer = torch.optim.Adam(model_nn.parameters(), lr=0.001)
nn_losses = []

for step in range(3000):
    ix = torch.randint(0, len(X_ctx3_train), (4096,))
    x_batch = encode_context(X_ctx3_train[ix], vocab_size)
    logits = model_nn(x_batch)
    loss = F.cross_entropy(logits, Y_ctx3_train[ix])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    nn_losses.append(loss.item())
    if step % 300 == 0:
        print(f"Стъпка {step:4d}: loss = {loss.item():.4f}")

In [ ]:
# Оценяване — encode_context за one-hot вход
with torch.no_grad():
    total_loss = 0.0
    correct = 0
    for i in range(0, len(X_ctx3_test), 4096):
        xb = encode_context(X_ctx3_test[i:i+4096], vocab_size)
        yb = Y_ctx3_test[i:i+4096]
        logits = model_nn(xb)
        total_loss += F.cross_entropy(logits, yb, reduction='sum').item()
        correct += (logits.argmax(1) == yb).sum().item()
    nn_test_loss = total_loss / len(X_ctx3_test)
    nn_test_acc = correct / len(X_ctx3_test)

# Резултати от Блок 5 (линеен контекст=3)
lin_ctx3_params, lin_ctx3_loss, lin_ctx3_acc = results[3]

print(f"{'Модел':<30} {'Параметри':>10} {'Test loss':>10} {'Accuracy':>10}")
print("-" * 64)
print(f"{'Линеен контекст=3 (Блок 5)':<30} {lin_ctx3_params:>10,} {lin_ctx3_loss:>10.4f} {lin_ctx3_acc:>10.1%}")
print(f"{'NN контекст=3 (Блок 7)':<30} {total_params:>10,} {nn_test_loss:>10.4f} {nn_test_acc:>10.1%}")
print()
print("→ Същите данни, същият контекст, но нелинейността чупи тавана!")
print()
print("=== Генериран текст (NN контекст=3) ===")
print(generate_context(model_nn, char_to_idx, idx_to_char, context_size=3, one_hot=True))

---
## Блок 8: Embedding слоеве [Демо]

**Цел:** Да заменим one-hot кодирането с научени embedding вектори — и да видим, че получаваме сравними резултати с много по-малко входни измерения. Това е по същество архитектурата на Bengio et al. (2003).

### От one-hot към embedding

При one-hot кодиране: `x @ W` просто избира един ред от матрицата W.
Умножаваме вектор с много нули по голяма матрица — за да изберем един ред.

**Защо да умножаваме по огромен разреден вектор?**
Можем директно да вземем реда: `W[i]`. Точно това прави `nn.Embedding`.

Освен това: embedding размерността не е задължително V.
Можем да използваме много по-малко измерения — и те са *научени* от данните!

**Сравнение:**
- **One-hot контекст=3:** вход с размер $3 \times 40 = 120$ (почти всички нули)
- **Embedding контекст=3 (2D):** вход с размер $3 \times 2 = 6$ (плътни научени вектори)

In [ ]:
emb = nn.Embedding(vocab_size, 2)
idx = torch.tensor([char_to_idx['а'], char_to_idx['б'], char_to_idx[' ']])
vectors = emb(idx)
print(f"Embedding: {vocab_size} символа × 2 измерения")
print(f"Вектор на 'а': {vectors[0].data.numpy().round(4)}")
print(f"Вектор на 'б': {vectors[1].data.numpy().round(4)}")
print(f"Вектор на ' ': {vectors[2].data.numpy().round(4)}")
print()
print(f"Вместо one-hot вектор с {vocab_size} елемента — само 2 числа!")
print(f"За контекст=3: {context_size * vocab_size} → {context_size * 2} входни измерения")

In [ ]:
class ContextEmbNN(nn.Module):
    """Embedding невронна мрежа с контекстен прозорец — архитектурата на Bengio et al. (2003)."""
    def __init__(self, vocab_size, context_size=3, embed_size=2, hidden_size=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.hidden = nn.Linear(context_size * embed_size, hidden_size)
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):  # x: (B, context_size) — целочислени индекси
        emb = self.embedding(x)            # (B, context_size, embed_size)
        emb = emb.view(x.size(0), -1)      # (B, context_size * embed_size)
        h = torch.sigmoid(self.hidden(emb))
        return self.output(h)

model_emb2 = ContextEmbNN(vocab_size, context_size, embed_size=2)
params_emb2 = sum(p.numel() for p in model_emb2.parameters())
print(f"ContextEmbNN (embed=2): {params_emb2:,} параметъра")
print(f"  Embedding:  {vocab_size} × 2 = {vocab_size * 2}")
print(f"  Скрит слой: {context_size * 2} × 128 + 128 = {context_size * 2 * 128 + 128}")
print(f"  Изход:      128 × {vocab_size} + {vocab_size} = {128 * vocab_size + vocab_size}")
print()
print(model_emb2)

In [ ]:
torch.manual_seed(42)
model_emb2 = ContextEmbNN(vocab_size, context_size, embed_size=2)
optimizer = torch.optim.Adam(model_emb2.parameters(), lr=0.01)
emb2_losses = []

for step in range(3000):
    ix = torch.randint(0, len(X_ctx3_train), (4096,))
    logits = model_emb2(X_ctx3_train[ix])  # Директно индекси, без one-hot!
    loss = F.cross_entropy(logits, Y_ctx3_train[ix])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    emb2_losses.append(loss.item())
    if step % 300 == 0:
        print(f"Стъпка {step:4d}: loss = {loss.item():.4f}")

In [ ]:
emb2_test_loss, emb2_test_acc = evaluate_nn(model_emb2, X_ctx3_test, Y_ctx3_test, vocab_size)

print(f"Embedding NN (2D) — loss: {emb2_test_loss:.4f}, accuracy: {emb2_test_acc:.1%}")
print(f"NN one-hot (Блок 7) — loss: {nn_test_loss:.4f}, accuracy: {nn_test_acc:.1%}")
print()
print("=== Генериран текст (Embedding NN, 2D) ===")
print(generate_context(model_emb2, char_to_idx, idx_to_char, context_size=3))

In [ ]:
emb_weights = model_emb2.embedding.weight.detach()

plt.figure(figsize=(12, 10))
for i, char in enumerate(chars):
    x_coord = emb_weights[i, 0].item()
    y_coord = emb_weights[i, 1].item()
    plt.scatter(x_coord, y_coord, s=30, alpha=0.7, color='steelblue')
    label = repr(char)[1:-1]  # Без кавичките
    plt.annotate(label, (x_coord, y_coord), fontsize=7, alpha=0.85,
                 xytext=(3, 3), textcoords='offset points')

plt.title("Научени 2D embedding вектори (контекст=3, embed_size=2)", fontsize=13)
plt.xlabel("Измерение 1")
plt.ylabel("Измерение 2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Наблюдавайте: сходни символи (гласни, съгласни, препинателни знаци) клъстеризират!")

### По-голямо embedding: embed_size=16

Тази архитектура е по същество моделът на **Bengio et al. (2003)** — "A Neural Probabilistic Language Model".
Всеки символ от контекста получава свой embedding вектор, те се конкатенират и минават през скрит слой.

С 2D embeddings можем да визуализираме, но нямаме достатъчно капацитет.
Нека опитаме с **embed_size=16** — по-близо до реалните модели.

In [ ]:
torch.manual_seed(42)
model_emb16 = ContextEmbNN(vocab_size, context_size, embed_size=16)
params_emb16 = sum(p.numel() for p in model_emb16.parameters())
optimizer = torch.optim.Adam(model_emb16.parameters(), lr=0.01)

print(f"ContextEmbNN (embed=16): {params_emb16:,} параметъра")
print(f"  Вход на скрития слой: {context_size} × 16 = {context_size * 16}")
print()

for step in range(3000):
    ix = torch.randint(0, len(X_ctx3_train), (4096,))
    logits = model_emb16(X_ctx3_train[ix])
    loss = F.cross_entropy(logits, Y_ctx3_train[ix])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 300 == 0:
        print(f"Стъпка {step:4d}: loss = {loss.item():.4f}")

emb16_test_loss, emb16_test_acc = evaluate_nn(model_emb16, X_ctx3_test, Y_ctx3_test, vocab_size)
print(f"\nEmbedding NN (16D) — loss: {emb16_test_loss:.4f}, accuracy: {emb16_test_acc:.1%}")

In [ ]:
print(f"{'Модел':<35} {'Параметри':>10} {'Test loss':>10} {'Accuracy':>10}")
print("-" * 69)
print(f"{'Случайно гадаене':<35} {'—':>10} {math.log(vocab_size):>10.4f} {1/vocab_size:>10.1%}")
print(f"{'Counting bigram (Л1)':<35} {'—':>10} {counting_loss:>10.4f} {counting_acc:>10.1%}")
print(f"{'Линеен bigram (Блок 4)':<35} {'1,640':>10} {test_loss:>10.4f} {test_acc:>10.1%}")
lin3_p, lin3_l, lin3_a = results[3]
print(f"{'Линеен контекст=3 (Блок 5)':<35} {lin3_p:>10,} {lin3_l:>10.4f} {lin3_a:>10.1%}")
print(f"{'NN контекст=3 (one-hot)':<35} {total_params:>10,} {nn_test_loss:>10.4f} {nn_test_acc:>10.1%}")
print(f"{'Embedding NN контекст=3 (2D)':<35} {params_emb2:>10,} {emb2_test_loss:>10.4f} {emb2_test_acc:>10.1%}")
print(f"{'Embedding NN контекст=3 (16D)':<35} {params_emb16:>10,} {emb16_test_loss:>10.4f} {emb16_test_acc:>10.1%}")
print()
print("Контекстът помага → нелинейността помага още повече → embeddings го правят ефективно.")

---
## Блок 9: Обобщение и дискусия [Дискусия]

**Цел:** Да консолидираме наученото и да обсъдим ограниченията и бъдещите посоки.

### Въпроси за дискусия

1. **Какво ще се случи ако увеличим контекста от 3 на 10 или 20 символа? Какви проблеми ще възникнат?**
   Помислете за: размер на входа, брой параметри, скорост на обучение, и дали конкатенирането на вектори е добра стратегия за дълъг контекст.
   *(Подсказка: в Лекция 4 ще видим как механизмът на вниманието решава този проблем.)*

2. **Работим на ниво символи.** Реалните модели ползват *токени* — части от думи.
   Защо? Какви са предимствата и недостатъците?

3. **Какво означава ако два символа имат близки embedding вектори?**
   Дайте пример с конкретни символи от визуализацията в Блок 8.

4. **GPT-2 има 124M параметъра. Нашият модел — няколко хиляди.**
   Какво позволява мащабът? Само повече памет ли е?

---

### Поглед напред

| Лекция | Тема | Връзка с днес |
|--------|------|----------------|
| **Л3** | Токенизация | Как се строи речникът на реален модел |
| **Л4** | Механизъм на вниманието | Как моделът избира кой контекст е важен (вместо да конкатенира всичко) |
| **Л5-6** | Трансформатор и обучение | Пълната архитектура на GPT-2 |

## Обобщение

1. **Линейният модел** представя символите като one-hot вектори и учи матрица W чрез градиентно спускане — вместо да брои.

2. **Градиентното спускане** работи в четири стъпки: forward pass → изчисляване на loss → backward pass → обновяване на теглата.

3. **Loss функцията** $-\log(P)$ е голяма при лоши предсказвания и малка при добри — тя е нашата мярка за качество.

4. **Линейният модел преоткри броенето** — softmax(W) ≈ bigram таблицата. Повече контекст помага, но **линейният модел удря таван**.

5. **Нелинейната активационна функция** (sigmoid) позволява на невронната мрежа да улови зависимости, които линейният модел не може. Със същия контекст=3 — нелинейността чупи тавана.

6. **Embedding слоеве** (nn.Embedding) заменят разредените one-hot вектори с компактни научени вектори — правят модела ефективен и разкриват структура (сходни символи → близки вектори).

7. **Архитектурата на Bengio et al. (2003):** embedding + скрит слой + softmax — е основата на невронните езикови модели. Следващата стъпка е механизмът на вниманието (Лекция 4).

## Ресурси

- **Andrej Karpathy — "makemore" серия (YouTube)**
  Части 1 и 2 покриват точно това, което построихме днес. Силно препоръчително за преглед.
  https://www.youtube.com/playlist?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ

- **Andrej Karpathy — "Let's build GPT" (YouTube)**
  Bigram моделът е отправната точка на тази лекция.
  https://www.youtube.com/watch?v=kCc8FmEb1nY

- **Bengio et al. (2003) — "A Neural Probabilistic Language Model"**
  Оригиналната статия с архитектурата от Блок 5 (Bengio диаграмата).
  https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf

- **3Blue1Brown — "But what is a neural network?" и "Gradient descent"**
  Отлични визуални обяснения на невронни мрежи и оптимизация.
  https://www.3blue1brown.com/topics/neural-networks

- **PyTorch tutorials — Tensors и Autograd**
  Официалната документация за основните концепции, използвани днес.
  https://pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html
  https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html

- **nanoGPT (GitHub: karpathy/nanoGPT)**
  Репозиторият, който ще разглеждаме в следващите лекции.
  https://github.com/karpathy/nanoGPT